In [9]:
import csv
import glob
from pathlib import Path

from nautilus_trader.model.data import Bar, BarType
from nautilus_trader.model.objects import Price, Quantity
from nautilus_trader.persistence.catalog import ParquetDataCatalog

# 纳秒级的合法时间范围（2017-07-14 ~ 2050-01-01）
MIN_TS_NS = 1500000000000 * 1_000_000    # 毫秒 -> 纳秒
MAX_TS_NS = 2524608000000 * 1_000_000

def ts_to_nanos(timestamp_str: str):
    """自动识别毫秒、微秒或纳秒，统一转为纳秒"""
    ts = int(timestamp_str)
    if ts <= 0:
        return None
    # 根据位数判断单位
    if 1e18 <= ts < 1e19:          # 19 位 → 纳秒
        return ts
    elif 1e15 <= ts < 1e16:        # 16 位 → 微秒
        return ts * 1000
    elif 1e12 <= ts < 1e13:        # 13 位 → 毫秒
        return ts * 1_000_000
    else:
        return None                # 异常位数，丢弃

def import_binance_csv_to_catalog():
    csv_dir = Path(r"D:\GitHub\Rust_trading\binance_downloader\binance_klines_data\BTCUSDT\1m")
    catalog_path = Path("./nautilus_data_catalog")
    
    catalog = ParquetDataCatalog(catalog_path)
    bar_type = BarType.from_str("BTCUSDT.BINANCE-1-MINUTE-LAST-EXTERNAL")
    
    all_csv_files = glob.glob(str(csv_dir / "*.csv"))
    if not all_csv_files:
        print("❌ 找不到 CSV 文件，请检查路径是否正确！")
        return
        
    print(f"📂 找到 {len(all_csv_files)} 个 CSV 文件，正在读取...")
    
    bars = []
    skipped = 0
    for file in all_csv_files:
        with open(file, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            for row in reader:
                if not row or not row[0].strip().isdigit():
                    continue
                if len(row) < 6:      # 至少需要 OHLCV
                    continue

                try:
                    ts_ns = ts_to_nanos(row[0])
                    if ts_ns is None or ts_ns < MIN_TS_NS or ts_ns > MAX_TS_NS:
                        print(f"⚠️ 跳过异常时间戳 {row[0]} (行: {row[:6]}...)")
                        skipped += 1
                        continue

                    bar = Bar(
                        bar_type=bar_type,
                        open=Price.from_str(row[1]),
                        high=Price.from_str(row[2]),
                        low=Price.from_str(row[3]),
                        close=Price.from_str(row[4]),
                        volume=Quantity.from_str(row[5]),
                        ts_event=ts_ns,
                        ts_init=ts_ns,
                    )
                    bars.append(bar)
                except Exception as e:
                    print(f"❌ 无法解析行 {row[:6]}...，错误: {e}")
                    skipped += 1
                    continue

    print(f"⚙️ 成功转换 {len(bars)} 根 K 线，跳过 {skipped} 行异常数据")
    catalog.write_data(bars)
    print(f"✅ 入库完成！目录: {catalog_path.resolve()}")

if __name__ == "__main__":
    import_binance_csv_to_catalog()

📂 找到 6 个 CSV 文件，正在读取...
⚙️ 成功转换 8640 根 K 线，跳过 0 行异常数据
✅ 入库完成！目录: D:\GitHub\Rust_trading\nautilus_data_catalog


In [10]:
from nautilus_trader.model.data import Bar
from nautilus_trader.model.enums import OrderSide
from nautilus_trader.trading.strategy import Strategy, StrategyConfig

class MyFirstStrategyConfig(StrategyConfig):
    # 这里定义策略的参数，比如均线周期
    fast_ma: int = 10
    slow_ma: int = 20

class MyFirstStrategy(Strategy):
    def __init__(self, config: MyFirstStrategyConfig):
        super().__init__(config)
        # 初始化变量和指标

    def on_start(self):
        # 策略启动时的逻辑，比如订阅行情
        self.request_bars(self.instrument_id)

    def on_bar(self, bar: Bar):
        # 核心业务逻辑：每次收到一根新的 1分钟 K线，这段代码就会被执行
        # 你可以在这里计算指标、判断多空并发送订单
        pass

In [13]:
import nautilus_trader.indicators as ind
# 查看 indicators 子模块列表
print(dir(ind))

['AdaptiveMovingAverage', 'ArcherMovingAveragesTrends', 'AroonOscillator', 'AverageTrueRange', 'Bias', 'BollingerBands', 'CandleBodySize', 'CandleDirection', 'CandleSize', 'CandleWickSize', 'ChandeMomentumOscillator', 'CommodityChannelIndex', 'DirectionalMovement', 'DonchianChannel', 'DoubleExponentialMovingAverage', 'EfficiencyRatio', 'ExponentialMovingAverage', 'FuzzyCandle', 'FuzzyCandlesticks', 'HullMovingAverage', 'IchimokuCloud', 'Indicator', 'KeltnerChannel', 'KeltnerPosition', 'KlingerVolumeOscillator', 'LinearRegression', 'MovingAverage', 'MovingAverageConvergenceDivergence', 'MovingAverageFactory', 'MovingAverageType', 'OnBalanceVolume', 'Pressure', 'PsychologicalLine', 'RateOfChange', 'RelativeStrengthIndex', 'RelativeVolatilityIndex', 'SimpleMovingAverage', 'SpreadAnalyzer', 'Stochastics', 'StochasticsDMethod', 'Swings', 'VariableIndexDynamicAverage', 'VerticalHorizontalFilter', 'VolatilityRatio', 'VolumeWeightedAveragePrice', 'WeightedMovingAverage', 'WilderMovingAverage',

In [15]:
import nautilus_trader.backtest as bt
print(dir(bt))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'config', 'data_client', 'engine', 'execution_client', 'models', 'modules', 'node', 'node_builder', 'results']


In [17]:
import inspect
from nautilus_trader.backtest.config import BacktestDataConfig
print(inspect.signature(BacktestDataConfig))

(catalog_path: str, data_cls: str, catalog_fs_protocol: str | None = None, catalog_fs_storage_options: dict | None = None, catalog_fs_rust_storage_options: dict | None = None, instrument_id: nautilus_trader.model.identifiers.InstrumentId | None = None, start_time: str | int | None = None, end_time: str | int | None = None, filter_expr: str | None = None, client_id: str | None = None, metadata: dict | typing.Any | None = None, bar_spec: str | None = None, instrument_ids: list[str] | None = None, bar_types: list[str] | None = None, optimize_file_loading: bool = False)


In [20]:
import inspect
from nautilus_trader.backtest.config import BacktestEngineConfig
print(inspect.signature(BacktestEngineConfig))

(environment: nautilus_trader.common.Environment = <Environment.BACKTEST: 'backtest'>, trader_id: nautilus_trader.model.identifiers.TraderId = 'BACKTESTER-001', instance_id: nautilus_trader.core.uuid.UUID4 | None = None, cache: nautilus_trader.cache.config.CacheConfig | None = CacheConfig(database=None, encoding='msgpack', timestamps_as_iso8601=False, persist_account_events=True, buffer_interval_ms=None, bulk_read_batch_size=None, use_trader_prefix=True, use_instance_id=False, flush_on_start=False, drop_instruments_on_reset=False, tick_capacity=10000, bar_capacity=10000), message_bus: nautilus_trader.common.config.MessageBusConfig | None = None, data_engine: nautilus_trader.data.config.DataEngineConfig | None = DataEngineConfig(time_bars_interval_type='left-open', time_bars_timestamp_on_close=True, time_bars_skip_first_non_full_bar=False, time_bars_build_with_no_updates=True, time_bars_origin_offset=None, time_bars_build_delay=0, validate_data_sequence=False, buffer_deltas=False, emit_

In [24]:
import inspect
from nautilus_trader.trading.config import ImportableStrategyConfig
print(inspect.signature(ImportableStrategyConfig))

(strategy_path: str, config_path: str, config: dict[str, typing.Any])


In [ ]:
import pandas as pd
from pathlib import Path

from nautilus_trader.model.data import Bar, BarType
from nautilus_trader.model.identifiers import InstrumentId, Venue
from nautilus_trader.trading.config import ImportableStrategyConfig  # 新增导入
from nautilus_trader.trading.config import ImportableStrategyConfig  # 新增导入
from nautilus_trader.model.enums import OrderSide
from nautilus_trader.model.objects import Money, Quantity
from nautilus_trader.indicators import ExponentialMovingAverage    # ✅ 已确认的路径
from nautilus_trader.trading.strategy import Strategy, StrategyConfig
from nautilus_trader.backtest.node import BacktestNode
from nautilus_trader.backtest.config import (                     # ✅ 修正后
    BacktestVenueConfig,
    BacktestDataConfig,
    BacktestEngineConfig,
    BacktestRunConfig,
)
from nautilus_trader.persistence.catalog import ParquetDataCatalog

# --------------------------- 策略配置 ---------------------------
class EMACrossConfig(StrategyConfig):
    instrument_id: str = "BTCUSDT.BINANCE"
    bar_type: str = "BTCUSDT.BINANCE-1-MINUTE-LAST-EXTERNAL"   # 必须与数据匹配
    fast_period: int = 10
    slow_period: int = 20
    trade_size: float = 0.1

# --------------------------- 策略主体 ---------------------------
class EMACrossStrategy(Strategy):
    def __init__(self, config: EMACrossConfig):
        super().__init__(config)
        self.instrument_id = InstrumentId.from_str(config.instrument_id)
        self.bar_type = BarType.from_str(config.bar_type)
        self.trade_size = config.trade_size

        self.fast_ema = ExponentialMovingAverage(config.fast_period)
        self.slow_ema = ExponentialMovingAverage(config.slow_period)

        self.cross_above = False
        self.cross_below = False

    def on_start(self):
        # 订阅正确的 Bar 类型
        self.subscribe_bars(self.bar_type)
        self.log.info("🚀 双均线策略启动")

    def on_bar(self, bar: Bar):
        # 更新指标
        self.fast_ema.update(bar.close.as_double())
        self.slow_ema.update(bar.close.as_double())

        if not self.fast_ema.initialized or not self.slow_ema.initialized:
            return

        fast_val = self.fast_ema.value
        slow_val = self.slow_ema.value

        # 金叉：买入（平空头、开多头）
        if fast_val > slow_val and not self.cross_above:
            self.cross_above = True
            self.cross_below = False
            # 先平空头（如果有）
            self.flatten_position(self.instrument_id)
            # 开多
            self.submit_order(
                self.order_factory.market(
                    instrument_id=self.instrument_id,
                    order_side=OrderSide.BUY,
                    quantity=self.instrument.make_qty(self.trade_size),
                )
            )
            self.log.info(f"📈 金叉买入 {self.trade_size}")

        # 死叉：卖出（平多头、开空头）
        elif fast_val < slow_val and not self.cross_below:
            self.cross_below = True
            self.cross_above = False
            # 先平多头
            self.flatten_position(self.instrument_id)
            # 开空
            self.submit_order(
                self.order_factory.market(
                    instrument_id=self.instrument_id,
                    order_side=OrderSide.SELL,
                    quantity=self.instrument.make_qty(self.trade_size),
                )
            )
            self.log.info(f"📉 死叉卖出 {self.trade_size}")

    @property
    def instrument(self):
        # 辅助方法，避免重复查找
        return self.cache.instrument(self.instrument_id)

# --------------------------- 回测启动 ---------------------------

def run_backtest():
    catalog_path = Path("./nautilus_data_catalog")
    catalog = ParquetDataCatalog(catalog_path)

    venue = Venue("BINANCE")
    venue_config = BacktestVenueConfig(
        name="BINANCE",
        oms_type="NETTING",
        account_type="MARGIN",
        base_currency="USDT",
        starting_balances=[Money.from_str("100000.00 USDT")],
    )

    bar_type = BarType.from_str("BTCUSDT.BINANCE-1-MINUTE-LAST-EXTERNAL")

    data_config = BacktestDataConfig(
        catalog_path=str(catalog_path),
        data_cls="Bar",
        instrument_id="BTCUSDT.BINANCE",
        bar_types=[str(bar_type)],
    )

    # 使用字符串路径 + 字典的方式配置策略
    strategy_import = ImportableStrategyConfig(
        strategy_path="__main__:EMACrossStrategy",   # 脚本主模块中的策略类
        config_path="__main__:EMACrossConfig",       # 脚本主模块中的配置类
        config={                                      # 配置参数字典
            "instrument_id": "BTCUSDT.BINANCE",
            "bar_type": "BTCUSDT.BINANCE-1-MINUTE-LAST-EXTERNAL",
            "fast_period": 10,
            "slow_period": 20,
            "trade_size": 0.1,
        },
    )

    config = BacktestRunConfig(
        engine=BacktestEngineConfig(strategies=[strategy_import]),
        venues=[venue_config],
        data=[data_config],
    )

    node = BacktestNode(configs=[config])

    print("⚙️ 开始回放历史数据...")
    results = node.run()
    print("\n✅ 回测完成！")
    return results

if __name__ == "__main__":
    results = run_backtest()

TypeError: Unexpected keyword argument 'strategy'